# Cartografía antigénica: reimplementación

Ajuste del mapa 2D a partir de la matriz de títulos de Katzelnick, en lugar de usar las coordenadas publicadas. Esto da un motor validable y, más adelante, la posibilidad de colocar secuencias nuevas con los sueros fijos.

Referencia del método: Smith et al. (2004), *Science* 305:371–376.

In [ ]:
# ===========================================================================
# Cartografia antigenica: reimplementacion del metodo de Smith et al. (2004),
# que es el que usan Katzelnick et al. (2015, 2021) para construir sus mapas.
#
# No es un MDS corriente. Tres cosas lo distinguen:
#
#   1. Las distancias son VIRUS-SUERO, no virus-virus. Por eso los sueros son
#      puntos del mapa: son las 36 filas 'SR' de 2d-antigenic-map.txt.
#   2. La distancia objetivo se mide contra el maximo de cada suero:
#          d_ij = base_j - log2(titulo_ij),   base_j = max log2 titulo del suero j
#      Una unidad antigenica (AU) = una dilucion al doble.
#   3. Un titulo '<X' NO afirma un valor: afirma una COTA INFERIOR de distancia.
#      Entra con perdida unilateral, que penaliza solo si el punto queda
#      demasiado CERCA. Son 2991 de 8643 titulaciones (35 %): tratarlas como
#      numeros -lo que hace la convencion X/2 de augur- sesga el mapa.
#
#        estres = SUM_medidos    (||x_i - s_j|| - d_ij)^2
#               + SUM_censurados max(0, d_ij - ||x_i - s_j||)^2
#
# Convenciones VERIFICADAS contra el mapa publicado, no asumidas:
#   - El umbral de '<X' es log2(X), no log2(X/2). Con X/2 el propio mapa
#     publicado violaria el 49 % de sus censuras; con X, solo el 14 % (3D).
#   - La base por columna incluye las censuradas. Sin eso, los tres sueros
#     cuyas celdas son TODAS censuradas quedan fuera de la perdida y flotan.
# ===========================================================================

from scipy.optimize import minimize   # la celda de imports solo trae least_squares


def read_titer_matrix(path):
    """Lee la matriz de titulos de Katzelnick.

    Devuelve (viruses, sera, ai, sj, d, is_cens, basej):
    ai/sj indexan el virus y el suero de cada titulacion, d es la distancia
    objetivo en AU, is_cens marca las celdas '<X'.
    """
    lines = [l.rstrip("\n") for l in open(path) if l.strip()]
    sera = lines[0].split()
    viruses, rec = [], []
    for i, l in enumerate(lines[1:]):
        p = l.split()
        viruses.append(p[0])
        for j, v in enumerate(p[1:]):
            if v == "*":                       # ausente: se descarta, no se imputa
                continue
            if v.startswith("<"):
                rec.append((i, j, np.log2(float(v[1:])), True))
            elif re.fullmatch(r"\d+(\.\d+)?", v):
                rec.append((i, j, np.log2(float(v)), False))

    ai = np.array([r[0] for r in rec])
    sj = np.array([r[1] for r in rec])
    t = np.array([r[2] for r in rec])
    is_cens = np.array([r[3] for r in rec])

    basej = np.full(len(sera), -np.inf)
    np.maximum.at(basej, sj, t)                # incluye censuradas: ver cabecera
    return viruses, sera, ai, sj, basej[sj] - t, is_cens, basej


def _acmap_stress(p, n_ag, n_sr, ndim, ai, sj, d, is_cens):
    """Estres y gradiente analitico. Devolverlos juntos evita recomputar."""
    X = p[:n_ag * ndim].reshape(n_ag, ndim)
    S = p[n_ag * ndim:].reshape(n_sr, ndim)
    diff = X[ai] - S[sj]
    dist = np.maximum(np.sqrt((diff ** 2).sum(1)), 1e-9)

    e = dist - d
    e = np.where(is_cens & (e > 0), 0.0, e)    # censurado y ya lejos -> sin costo

    g = (2.0 * e / dist)[:, None] * diff
    gX = np.zeros_like(X)
    gS = np.zeros_like(S)
    np.add.at(gX, ai, g)
    np.add.at(gS, sj, -g)
    return float((e ** 2).sum()), np.concatenate([gX.ravel(), gS.ravel()])


def fit_acmap(ai, sj, d, is_cens, n_ag, n_sr, ndim=2, n_starts=60, seed=0,
              verbose=False):
    """Multiples arranques aleatorios; se queda el de menor estres.

    El paisaje tiene minimos locales: en esta matriz el estres va de ~5000 a
    ~10000 segun el arranque, asi que un solo arranque no sirve.
    """
    rng = np.random.default_rng(seed)
    scale = float(np.median(d[~is_cens]))
    best = None
    for k in range(n_starts):
        p0 = rng.normal(scale=scale, size=(n_ag + n_sr) * ndim)
        r = minimize(_acmap_stress, p0, jac=True, method="L-BFGS-B",
                     args=(n_ag, n_sr, ndim, ai, sj, d, is_cens),
                     options=dict(maxiter=20000, maxfun=50000,
                                  ftol=1e-12, gtol=1e-10))
        if best is None or r.fun < best.fun:
            best = r
        if verbose:
            print(f"  arranque {k+1:3d}/{n_starts}  estres={r.fun:9.2f}  "
                  f"mejor={best.fun:9.2f}")
    return (best.x[:n_ag * ndim].reshape(n_ag, ndim),
            best.x[n_ag * ndim:].reshape(n_sr, ndim),
            best.fun)


def procrustes_align(ref, mov):
    """Alinea 'mov' sobre 'ref': traslacion, rotacion y reflexion.

    SIN escalado, a proposito: en un mapa antigenico una unidad es una dilucion
    al doble, asi que reescalar destruiria la interpretacion. Un mapa antigenico
    solo esta determinado salvo estas tres transformaciones.
    """
    rc, mc = ref - ref.mean(0), mov - mov.mean(0)
    U, _, Vt = np.linalg.svd(mc.T @ rc)
    return mc @ (U @ Vt) + ref.mean(0)


print("Motor de cartografia cargado.")


## Validación contra el mapa publicado

In [ ]:
# ===========================================================================
# Validacion: la reimplementacion reproduce el mapa publicado?
#
# Este control no tiene equivalente en la ruta de augur: aqui existe una
# salida publicada del mismo procedimiento, asi que el motor se puede
# contrastar en vez de darlo por bueno.
#
# Se reporta por separado los puntos BIEN y MAL determinados. Mezclarlos
# oculta la calidad real: un suero cuyas celdas son todas censuradas no tiene
# posicion identificable y su desviacion no mide nada del metodo.
# ===========================================================================

NDIM = 2
N_STARTS = 60          # ~2-3 min; con 20 el estres queda en ~5100 en vez de ~4983
MIN_MEDIDAS = 4        # umbral para considerar un punto bien determinado

viruses, sera, ai, sj, d_tgt, is_cens, basej = read_titer_matrix(TITERS_KATZ)
n_ag, n_sr = len(viruses), len(sera)
print(f"{n_ag} virus x {n_sr} sueros | {len(d_tgt)} titulaciones "
      f"({(~is_cens).sum()} medidas, {is_cens.sum()} censuradas, "
      f"{n_ag*n_sr - len(d_tgt)} ausentes)")

X_fit, S_fit, stress = fit_acmap(ai, sj, d_tgt, is_cens, n_ag, n_sr,
                                 ndim=NDIM, n_starts=N_STARTS, seed=2)

dist_fit = np.linalg.norm(X_fit[ai] - S_fit[sj], axis=1)
med = ~is_cens
res = dist_fit[med] - d_tgt[med]
viol = dist_fit[is_cens] < d_tgt[is_cens]

# --- mapa publicado, en el mismo orden
pub_ag = mapa_ag.set_index("strainID")[["coord1", "coord2"]]
pub = np.vstack([pub_ag.loc[viruses].to_numpy(), pub_ag.loc[sera].to_numpy()])
mine = np.vstack([X_fit, S_fit])
aligned = procrustes_align(pub, mine)
dev = np.linalg.norm(aligned - pub, axis=1)

# el mapa publicado medido con su propio criterio: la referencia a batir
d_pub = np.linalg.norm(pub[:n_ag][ai] - pub[n_ag:][sj], axis=1)
rmse_pub = np.sqrt(((d_pub[med] - d_tgt[med]) ** 2).mean())

n_med = np.concatenate([np.bincount(ai[med], minlength=n_ag),
                        np.bincount(sj[med], minlength=n_sr)])
good = n_med >= MIN_MEDIDAS

print("\n" + "=" * 78)
print("1. AJUSTE PROPIO")
print("=" * 78)
print(f"estres = {stress:.2f}")
print(f"RMSE sobre celdas medidas : {np.sqrt((res**2).mean()):.3f} AU  "
      f"(sesgo {res.mean():+.3f})")
print(f"  mismo numero, mapa publicado: {rmse_pub:.3f} AU")
print(f"censuras violadas         : {viol.sum()}/{is_cens.sum()} "
      f"({100*viol.mean():.1f} %)")
print("\nUna AU = una dilucion al doble. El RMSE del mapa publicado (~0.95 AU)")
print("es el estres intrinseco del metodo y ronda el ruido del ensayo (~1 log2):")
print("es una COTA DURA para cualquier cosa que se proyecte despues sobre el mapa.")

print("\n" + "=" * 78)
print("2. PROCRUSTES CONTRA EL MAPA PUBLICADO")
print("=" * 78)
print(f"{'grupo':22s} {'n':>5s} {'RMSD':>7s} {'mediana':>8s} {'p95':>7s}")
for lbl, sel in (("todos", np.ones(len(dev), bool)),
                 (f"bien determinados", good),
                 ("mal determinados", ~good)):
    if sel.sum():
        print(f"{lbl:22s} {sel.sum():5d} {np.sqrt((dev[sel]**2).mean()):7.3f} "
              f"{np.median(dev[sel]):8.3f} {np.percentile(dev[sel], 95):7.3f}")
gv, gs = good[:n_ag], good[n_ag:]
print(f"{'  de ellos virus':22s} {gv.sum():5d} "
      f"{np.sqrt((dev[:n_ag][gv]**2).mean()):7.3f}")
print(f"{'  de ellos sueros':22s} {gs.sum():5d} "
      f"{np.sqrt((dev[n_ag:][gs]**2).mean()):7.3f}")

print(f"\nPuntos con menos de {MIN_MEDIDAS} titulaciones medidas:")
for k in np.argsort(-dev * ~good)[: (~good).sum()]:
    nom = viruses[k] if k < n_ag else sera[k - n_ag]
    print(f"  {'virus' if k < n_ag else 'SUERO'} {nom:42s} "
          f"dev={dev[k]:6.2f} AU  medidas={n_med[k]}")
print("\nLos sueros con 0 medidas NO son identificables: todas sus celdas son")
print("censuradas, de modo que sus restricciones son cotas inferiores vacuas.")
print("Sus coordenadas publicadas tampoco significan nada. Excluirlos.")

print("\n" + "=" * 78)
print("3. VEREDICTO")
print("=" * 78)
rmsd_good = np.sqrt((dev[good] ** 2).mean())
print(f"Error de reproduccion sobre los {good.sum()} puntos determinados: "
      f"{rmsd_good:.3f} AU")
print(f"Estres intrinseco del mapa publicado                      : {rmse_pub:.3f} AU")
print("REPRODUCE el mapa publicado." if rmsd_good < rmse_pub else
      "NO lo reproduce: revisar convenciones antes de seguir.")

# --- guardar
out = pd.DataFrame({
    "type":   ["AG"] * n_ag + ["SR"] * n_sr,
    "strainID": list(viruses) + list(sera),
    "coord1": aligned[:, 0], "coord2": aligned[:, 1],
    "n_medidas": n_med, "bien_determinado": good,
    "dev_vs_publicado": dev,
})
out.to_csv(ACMAP_FIT.format(NDIM), index=False)
print(f"\ncoordenadas -> {ACMAP_FIT.format(NDIM)}")


### Figura: reproducción del mapa

In [ ]:
# ===========================================================================
# Figura: mapa reajustado frente al publicado.
#
# Panel A/B: los dos mapas, alineados por Procrustes. Deben verse iguales.
# Panel C: desviacion por punto frente a cuantas titulaciones lo anclan. Es la
#          figura que justifica separar bien y mal determinados: lo que se
#          desvia es lo que carece de datos, no un sesgo del metodo.
# ===========================================================================

INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
sero_of = lambda s: int(re.search(r"(?:DENV|DEN)(\d)", s).group(1))
cols_ag = [SERO_COLORS[sero_of(v)] for v in viruses]
cols_sr = [SERO_COLORS[sero_of(s)] for s in sera]

fig = plt.figure(figsize=(15, 5.4))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.95], wspace=0.28)

# Limites comunes a A y B, fijados sobre los puntos determinados: si no, los
# sueros no identificables estiran un panel y los dos dejan de ser comparables.
_lo = np.minimum(pub[good].min(0), aligned[good].min(0)) - 1.0
_hi = np.maximum(pub[good].max(0), aligned[good].max(0)) + 1.0

for k, (coords, titulo) in enumerate((
        (pub, "A. Mapa publicado (Katzelnick)"),
        (aligned, "B. Reajustado desde los titulos"))):
    ax = fig.add_subplot(gs[0, k])
    ax.scatter(coords[:n_ag, 0], coords[:n_ag, 1], s=16, c=cols_ag,
               alpha=0.75, linewidths=0, zorder=2)
    ax.scatter(coords[n_ag:, 0], coords[n_ag:, 1], s=55, facecolors="none",
               edgecolors=cols_sr, linewidths=1.4, marker="s", zorder=3)
    ax.set_title(titulo, fontsize=11, color=INK)
    ax.set_aspect("equal")
    ax.set_xlim(_lo[0], _hi[0])
    ax.set_ylim(_lo[1], _hi[1])
    ax.set_xlabel("AU (1 unidad = dilucion al doble)", fontsize=9, color=INK_MUTED)

ax = fig.add_subplot(gs[0, 2])
_x = np.maximum(n_med, 0.5)      # los puntos con 0 medidas, visibles en escala log
_jit = np.random.default_rng(0).normal(1.0, 0.04, len(dev))
ax.scatter(_x[good] * _jit[good], dev[good], s=18, color="#2a78d6",
           alpha=0.55, linewidths=0, label=f">= {MIN_MEDIDAS} medidas")
ax.scatter(_x[~good] * _jit[~good], dev[~good], s=46, color="#eb6834",
           alpha=0.9, linewidths=0, label=f"< {MIN_MEDIDAS} medidas")
ax.set_xscale("log")
ax.axhline(rmse_pub, color=INK_MUTED, linestyle="--", linewidth=1.2)
ax.annotate(f"estres del mapa publicado ({rmse_pub:.2f} AU)",
            xy=(0.97, rmse_pub), xycoords=("axes fraction", "data"),
            ha="right", va="bottom", fontsize=8, color=INK_MUTED)
ax.set_yscale("log")
ax.set_xlabel("titulaciones medidas que anclan el punto", fontsize=9, color=INK_MUTED)
ax.set_ylabel("desviacion vs mapa publicado (AU)", fontsize=9, color=INK_MUTED)
ax.set_title("C. Lo que se desvia es lo que no tiene datos", fontsize=11, color=INK)
ax.legend(frameon=False, fontsize=8, loc="upper right")

for ax in fig.axes:
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(labelsize=8, colors=INK_MUTED)

handles = [plt.Line2D([], [], marker="o", linestyle="", color=SERO_COLORS[s],
                      label=f"DENV{s}") for s in sorted(SERO_COLORS)]
handles += [plt.Line2D([], [], marker="s", linestyle="", markerfacecolor="none",
                       markeredgecolor=INK_MUTED, label="suero")]
fig.legend(handles=handles, frameon=False, fontsize=9, ncol=5,
           loc="lower center", bbox_to_anchor=(0.36, -0.04))

_out = "../figures/acmap_reproduccion.png"
fig.savefig(_out, dpi=300, bbox_inches="tight")
print(f"figura -> {_out}")
plt.show()
